In [1]:
from scipy.ndimage import gaussian_filter
from scipy.signal import savgol_filter
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from shared_functions import mask_movie
import cv2
from skimage.draw import polygon2mask
from skimage.measure import find_contours

In [2]:
# Loading and processing HCP movies

def get_ttl_trace(timestamps_table, timestamps_table_names, timeorigin, timebinning=1):
    """
    Extracts the TTL trace from the movie specs.

    Args:
    - timestamps_table (np.ndarray): Timestamps table containing TTL data.
    - timestamps_table_names (np.ndarray): Column names for the timestamps table.
    - timeorigin (int): Starting index for the TTL signal.
    - timebinning (int, optional): Time binning factor for downsampling (default: 1).

    Returns:
    - np.ndarray: Extracted TTL signal.
    """
  
    # Get the column index for 'behavior_ttl'
    ttl_column = timestamps_table_names.index("behavior_ttl")

    # Extract the raw TTL signal starting from `timeorigin`
    ttl_signal_raw = timestamps_table[ttl_column, int(timeorigin):]

    # Apply binning if `timebinning` is greater than 1
    if timebinning > 1:
        # Ensure the length of ttl_signal_raw is divisible by timebinning
        trimmed_length = len(ttl_signal_raw) - (len(ttl_signal_raw) % timebinning)
        ttl_signal_raw = ttl_signal_raw[:trimmed_length]
        
        # Reshape, average over bins, and round the result
        ttl_signal = np.round(np.mean(ttl_signal_raw.reshape(-1, timebinning), axis=1))
    else:
        ttl_signal = ttl_signal_raw

    onsets = np.where(np.diff(ttl_signal) == 1)[0] 
    offsets = np.where(np.diff(ttl_signal) == -1)[0] 

    return onsets, offsets

def movie_path(mouse, date, file):
    """Returns the file path for a given movie recording."""
    return "N:/GEVI_Wave/Analysis/Visual/" + mouse + "/20" + str(date) + "/" + file + '/cG_unmixed_dFF.h5'

def mask_movie(movie, raw_mask, binning):
    """
    Applies a spatial mask to a movie to exclude non-brain regions.

    Args:
    - movie (numpy.ndarray): 3D array representing the movie (t, x, y).
    - raw_mask (numpy.ndarray): 2D mask array.
    - binning (float): Scaling factor for resizing the mask.

    Returns:
    - numpy.ndarray: The masked movie with non-brain regions removed.
    """
    # Resize the mask using bilinear interpolation
    mask = cv2.resize(raw_mask, (0, 0), fx=1/binning, fy=1/binning, interpolation=cv2.INTER_LINEAR)
    
    # Ensure mask has the same shape as the movie
    movie_size = movie.shape[1:3]  # Get spatial dimensions of the movie
    mask = mask[:movie_size[0], :movie_size[1]]  # Crop mask to match movie size
    mask = np.flip(mask, axis=0)
    mask = mask.astype(bool)         # Convert to boolean mask
    return movie * mask  # Apply mask

def mask_region(movie, raw_outlines, binning, spaceorigin, region, plot=False):
    """
    Applies a mask to a movie based on anatomical brain region outlines.

    Args:
    - movie (numpy.ndarray): 3D movie array (t, x, y).
    - raw_outlines (numpy.ndarray): 3D array of brain region outlines.
    - binning (float): Scaling factor for spatial binning.
    - spaceorigin (numpy.ndarray): 2D array of space origin coordinates.
    - region (str): Brain region to mask ('V1' or 'SSC').
    - plot (bool, optional): Whether to display the mask overlay on the first frame of the movie.

    Returns:
    - numpy.ndarray: The masked movie where only the selected region remains.
    """

    # Define the corresponding outline indices for the selected brain region
    if region == 'V1':
        indices=[37]
    elif region == 'SSC':
        indices=[3,11,13,15]
    else:
        print('Region must be V1 or SSC')

    spaceorigin = (spaceorigin - 1) / binning + 1  # Apply space origin transformation

    # Extract all outlines and scale them according to binning
    outlines_nums = np.arange(raw_outlines.shape[2]) 
    outlines = raw_outlines[:, :, outlines_nums] / binning

    outlines[:, 0, :] -= spaceorigin[1] - 1  # Adjust Y-coordinates
    outlines[:, 1, :] -= spaceorigin[0] - 1  # Adjust X-coordinates

    # Define the movie dimensions
    movie_shape = movie.shape[1:3]  # (height, width)
            
    if plot:
    # Plot the first movie frame
        plt.figure(figsize=(6, 6))
        plt.imshow(movie[0])  # Display movie frame
        plt.title(f"Movie with {region} Outline")

    total_mask = np.zeros(movie_shape, dtype=bool)

    for i in indices:
        # Extract the ROI outline
        outline = outlines[i, :, :]  # Shape (2, N)

        valid_indices = ~np.isnan(outline).any(axis=0)  # Find non-NaN indices
        x_coords = outline[1, valid_indices]
        y_coords = outline[0, valid_indices]

        # Create a mask using polygon2mask
        roi_mask = polygon2mask(movie_shape, np.column_stack((y_coords, x_coords)))
        roi_mask = np.flipud(roi_mask).astype(bool)
        total_mask |= roi_mask  # Any pixel belonging to at least one ROI remains unmasked

    # Apply mask to the movie
    movie_roi = movie * total_mask  # Broadcasting applies the mask to all frames

    # Find contours of the ROI mask
    contours = find_contours(total_mask, level=0.5)  # Extract the outline

    if plot:
        # Overlay the mask outline
        for contour in contours:
            plt.plot(contour[:, 1], contour[:, 0], color='red', linewidth=1.5)  # Plot contour in red
    plt.show()

    return movie_roi

def load_and_mask(path_movie, path_specs=None, mask_v1=False):
    """
    Loads a movie file, applies masking, and optionally extracts a specific brain region.

    Args:
    - mouse (str): Mouse identifier.
    - date (str or int): Experiment date in YYMMDD format.
    - file (str): Recording file identifier.
    - region (str, optional): Brain region to isolate ('V1' or 'SSC'). If None, the full movie is returned.

    Returns:
    - movie (numpy.ndarray): 3D array (t, x, y) representing the processed movie.
    - fps (float): Frames per second of the movie.
    """

    with h5py.File(path_movie, 'r') as mov_file:
        mov = mov_file['mov'][()]  

    if not path_specs:
        path_specs = path_movie
    with h5py.File(path_specs, 'r') as specs_file:
        specs = specs_file["specs"]

        fps = specs["fps"][()].squeeze() 
        raw_mask = specs["extra_specs"]["mask"][()].squeeze() 
        binning = specs["binning"][()].squeeze() 
        raw_outlines = specs["extra_specs"]["allenMapEdgeOutline"][()].squeeze() 
        spaceorigin = specs["spaceorigin"][()].squeeze()  

        timeorigin = specs["timeorigin"][()].squeeze() 
        timebinning = specs["timebinning"][()].squeeze() 
        timestamps_table = specs["extra_specs"]["timestamps_table"][()].squeeze()
        timestamps_table_names = specs["extra_specs"]["timestamps_table_names"][()].squeeze() 
        timestamps_table_names = b''.join(timestamps_table_names.flatten()).decode("utf-8").split(';')

    onsets, offsets = get_ttl_trace(timestamps_table, timestamps_table_names, timeorigin, timebinning)

    # Replace NaN values in the movie
    movie = np.nan_to_num(mov)
    movie = np.flip(movie, axis=1)

    # Apply the brain mask to remove non-brain regions
    movie = mask_movie(movie, raw_mask, binning)

    # If a specific brain region is provided, apply the corresponding region mask
    if mask_v1:
        movie = mask_region(movie, raw_outlines, binning, spaceorigin, 'V1', plot=False)

    return movie, fps, onsets, offsets

In [3]:
def analytic_signal_fft_nd(x, axis=-1):
    """
    FFT-based Hilbert transform to get analytic signal along a given axis.
    Works for real x with arbitrary extra dims.
    Returns complex analytic signal with same shape.
    """
    x = np.asarray(x)
    n = x.shape[axis]
    X = np.fft.fft(x, n=n, axis=axis)
    h = np.zeros(n, dtype=float)
    if n % 2 == 0:
        h[0] = 1.0
        h[n//2] = 1.0
        h[1:n//2] = 2.0
    else:
        h[0] = 1.0
        h[1:(n+1)//2] = 2.0
    # reshape h to broadcast along 'axis'
    shape = [1]*x.ndim
    shape[axis] = n
    H = h.reshape(shape)
    Z = np.fft.ifft(X * H, axis=axis)
    return Z

def gaussian_filter_nan(a, sigma, truncate=3.0, mode='reflect'):
    """
    NaN-aware Gaussian that (i) renormalizes by valid support and
    (ii) guarantees the output is NaN exactly where input is NaN.
    Works for (ny,nx) or (ny,nx,nt); use sigma like (sx,sy,0.0) for movies.
    """
    orig_valid = np.isfinite(a)
    a_filled   = np.where(orig_valid, a, 0.0).astype(float)

    # Constant padding avoids reflecting values across the mask boundary
    w = gaussian_filter(orig_valid.astype(float), sigma=sigma,
                        truncate=truncate, mode='constant', cval=0.0)
    b = gaussian_filter(a_filled, sigma=sigma,
                        truncate=truncate, mode='constant', cval=0.0)

    out = b / (w + 1e-12)
    # (i) keep NaNs where there was no valid support at all
    out[w < 1e-12] = np.nan
    # (ii) clamp to original mask: never “create” new pixels
    out[~orig_valid] = np.nan
    return out

def compute_phase(
    movie, Fs,
    sigma_px=1.0, truncate=3.0, mode='reflect',
    dx=1.0, dy=1.0
):
    """
    End-to-end analysis with the SAME methods as your current function:
      - Optional temporal bandpass (Butterworth, sosfiltfilt)
      - Analytic signal via analytic_signal_fft_nd
      - Amplitude & wrapped phase
      - Wrap-safe smoothing by normalizing to unit-magnitude complex, then Gaussian
      - Phase gradient via complex-product forward differences, sign-flipped (g = -∇φ)
      - Gradient magnitude

    Parameters
    ----------
    movie : (nt, ny, nx) real array
    Fs    : sampling rate (Hz)
    lo, hi, order : temporal bandpass parameters
    sigma_px, truncate, mode : spatial Gaussian params for complex smoothing
    dx, dy : pixel size (x,y)
    amp_mask : bool, apply your mask_low_amp() to z and amp
    bandpass : bool, apply bandpass before analytic signal
    identify_source : bool, compute argmax of divergence per frame for convenience
    gp : bool, whether to compute generalized phase

    Returns
    -------
    results : dict with keys:
        't', 'movie_bp', 'z', 'amp', 'phi', 'z_s',
        'gx', 'gy', 'grad_mag', 'div', 'curl', 'source'
    """
    nt, ny, nx = movie.shape
    t = np.arange(nt) / Fs

    # ---- 2) Analytic signal, amplitude, phase (unchanged) ----
    z = analytic_signal_fft_nd(movie, axis=0)
    amp = np.abs(z)

    # ---- Optional low-amp mask (unchanged) ----
    phi = np.angle(z)

    # ---- 3) Normalize amplitude ----
    eps = 1e-12
    
    # ---- Safe unit complex field: z_unit = z / amp ----
    # valid where both z and amp are finite and amp > eps
    valid = np.isfinite(z) & np.isfinite(amp) & (amp > eps)

    z_s = np.full_like(z, np.nan, dtype=np.complex128)     # preserves mask
    np.divide(z, amp, out=z_s, where=valid)             # no divide at invalids

    # ---- 4) Phase gradient via complex-product forward diffs (unchanged) ----
    # gx ~ angle( z(x+1,y) * conj(z(x,y)) ) / dx ; same for gy, then sign flip for g = -∇φ
    z_xp1 = np.roll(z_s, -1, axis=2)
    dphi_x = np.angle(z_xp1 * np.conj(z_s))
    gx = dphi_x / dx
    gx[:, :, -1] = np.nan  # invalidate forward edge

    z_yp1 = np.roll(z_s, -1, axis=1)
    dphi_y = np.angle(z_yp1 * np.conj(z_s))
    gy = dphi_y / dy
    gy[:, -1, :] = np.nan  # invalidate forward edge

    # Convention match: g = -∇φ
    gx, gy = -gx, -gy
    grad_mag = np.sqrt(gx**2 + gy**2)

    # take unit vectors of gradient
    eps = 1e-12
    gx = gx / (grad_mag + eps)
    gy = gy / (grad_mag + eps)
    
    # ---- Smooth the unit vector field ----
    sigma_px = 3
    u = gx + 1j * gy
    ur = gaussian_filter_nan(np.real(u), sigma=(sigma_px, sigma_px, 0.0), truncate=truncate)
    ui = gaussian_filter_nan(np.imag(u), sigma=(sigma_px, sigma_px, 0.0), truncate=truncate)
    u_s = ur + 1j*ui

    # ---- Safe renormalization: u_hat = u_s / |u_s| ----
    mag    = np.abs(u_s)
    validU = np.isfinite(u_s) & np.isfinite(mag) & (mag > eps)

    u_hat = np.full_like(u_s, np.nan, dtype=np.complex128)
    np.divide(u_s, mag, out=u_hat, where=validU)

    gx, gy = np.real(u_hat), np.imag(u_hat)

    return amp, phi, grad_mag, gx, gy

def mean_phase_gradient(
    gx, gy, clip, fps, 
    t_start_s, t_end_s, 
    x_range=(90,110), y_range=(150,170),
    weight=True,
    plot=False
    ):
    """
    Time/space average of phase-gradient unit vectors in a rectangular ROI,
    for data shaped (t, y, x). Optionally plots a gx frame with ROI overlay.

    Parameters
    ----------
    gx, gy : arrays shaped (t, y, x)
        Unit-vector components of the phase gradient.
    clip : array shaped (t, y, x)
        ΔF/F movie clip
    fps : float
        Frames per second for time indexing.
    t_start_s, t_end_s : float
        Time window in seconds (half-open [start, end)).
    x_range, y_range : tuple[int, int]
        Inclusive pixel index ranges for the ROI.
    weights : None or array shaped (t, y, x)
        Optional weights (e.g. amplitude) for averaging.
    plot : bool
        If True, show a gx frame with the ROI rectangle.

    Returns
    -------
    "theta_deg": angle (degrees),
    "R": mean resultant length (0–1)
    """
    # --- indices ---
    t0 = int(np.floor(t_start_s * fps))
    t1 = int(np.ceil(t_end_s * fps))
    t0 = max(0, min(t0, gx.shape[0]))
    t1 = max(0, min(t1, gx.shape[0]))

    y0, y1_inc = y_range
    x0, x1_inc = x_range
    ys = slice(y0, y1_inc + 1)
    xs = slice(x0, x1_inc + 1)

    gx_roi = gx[t0:t1, ys, xs]
    gy_roi = gy[t0:t1, ys, xs]
    valid = np.isfinite(gx_roi) & np.isfinite(gy_roi)

    if weight:
        weights = np.abs(clip)
        w = weights[t0:t1, ys, xs].astype(float)
        w[~valid | ~np.isfinite(w)] = 0.0
    else:
        w = np.ones_like(gx_roi)

    wx = w * gx_roi
    wy = w * gy_roi
    denom = np.sum(w)

    mx = np.sum(wx) / denom
    my = np.sum(wy) / denom
    R = np.hypot(mx, my)
    theta = np.arctan2(my, mx)
    theta_deg = np.degrees(theta)

    return float(theta_deg), float(R)

def _wrap_pi(a):
    """Wrap angles (radians) to (-pi, pi]."""
    return (a + np.pi) % (2*np.pi) - np.pi

def _time_to_idx(t_s, fps, n):
    """Clamp time (s) to frame index [0, n]."""
    i = int(np.round(t_s * fps))
    return max(0, min(n, i))

def roi_vectors_per_frame(
    gx, gy, clip, fps,
    x_range, y_range,
    weight=True,
):
    """
    Compute per-frame mean phase-gradient vector (mx_t, my_t) in ROI,
    then return theta_t (rad) and R_t in shape (t,).

    Parameters
    ----------
    gx, gy : (t, y, x) unit-vector fields
    clip   : (t, y, x) ΔF/F (for weights if weight=True)
    fps    : float
    x_range, y_range : inclusive pixel index ranges (x0, x1), (y0, y1)
    weight : if True, use |ΔF/F| as per-pixel weights

    Returns
    -------
    theta_t : (t,) radians, wrapped to (-pi, pi]
    R_t     : (t,) mean resultant length in [0, 1]
    valid_t : (t,) bool, True when denominator > 0
    """
    y0, y1_inc = y_range
    x0, x1_inc = x_range
    ys = slice(y0, y1_inc + 1)
    xs = slice(x0, x1_inc + 1)

    gx_roi = gx[:, ys, xs]
    gy_roi = gy[:, ys, xs]
    valid = np.isfinite(gx_roi) & np.isfinite(gy_roi)

    if weight:
        w = np.abs(clip[:, ys, xs]).astype(float)
        w[~valid | ~np.isfinite(w)] = 0.0
    else:
        w = np.where(valid, 1.0, 0.0)

    # sums across spatial dims -> (t,)
    wx = np.sum(w * gx_roi, axis=(1, 2))
    wy = np.sum(w * gy_roi, axis=(1, 2))
    denom = np.sum(w, axis=(1, 2))

    valid_t = denom > 0
    mx = np.zeros_like(wx, dtype=float)
    my = np.zeros_like(wy, dtype=float)
    mx[valid_t] = wx[valid_t] / denom[valid_t]
    my[valid_t] = wy[valid_t] / denom[valid_t]

    R_t = np.hypot(mx, my)
    theta_t = np.arctan2(my, mx)
    theta_t = _wrap_pi(theta_t)

    return theta_t, R_t, valid_t

def _roi_slices(x_range, y_range):
    """Return (ys, xs) slices for numpy indexing given half-open pixel ranges."""
    x0, x1 = x_range
    y0, y1 = y_range
    return slice(y0, y1), slice(x0, x1)

def _wrapsafe_dphi_dt(phi, fps):
    """
    Wrap-safe temporal phase derivative using the complex-exponent trick.

    Parameters
    ----------
    phi : array, shape (t, y, x)
        Wrapped phase (radians), per pixel.
    fps : float
        Sampling rate (Hz).

    Returns
    -------
    dphi_dt : array, shape (t-1, y, x)
        Wrap-safe temporal derivative of phase (rad/s) between consecutive frames.
        By convention for φ = k·r - ω t, ∂φ/∂t = -ω (negative).
    """
    # Δφ wrapped into (-π, π]
    dphi = np.angle(np.exp(1j * (phi[1:] - phi[:-1])))
    return dphi * fps

def per_frame_wave_traces(amp, phi, grad_mag, fps, x_range, y_range,
                          smooth_window_s=0.15, smooth_poly=2):
    """
    Compute per-frame, ROI-aggregated traces:
      - frequency f_t (Hz) from wrap-safe ∂φ/∂t
      - wavenumber k_t (rad/pixel) from |∇φ|
      - wavelength λ_t (pixel) via 2π / k_t
      - phase speed v_t (pixel/s) = f_t * λ_t (aligned with f_t)
      - mean Hilbert amplitude A_t within ROI
      (All final vectors are smoothed in time.)

    Notes
    -----
    * Update: unweighted ROI means; hard-coded k gate kmax=0.5 (no kmin).
    * Smoothing: Savitzky–Golay with ~0.15 s window (default) and poly=2.
    * Shapes:
        k_t, λ_t, A_t  -> length T
        f_t, v_t       -> length T, last element NaN (aligned to frame grid)

    Returns
    -------
    traces : dict with keys ['f_t','k_t','lambda_t','v_t','A_t']
    """

    def _odd(n):  # smallest odd >= n
        return n if (n % 2) else n + 1

    def _savgol_nan(x, wl, po):
        """Savitzky–Golay with NaN handling via linear interpolation."""
        x = np.asarray(x, dtype=float)
        mask = np.isfinite(x)
        if mask.sum() < (po + 2):  # not enough valid points
            return x
        xi = np.arange(x.size)
        x_fill = x.copy()
        x_fill[~mask] = np.interp(xi[~mask], xi[mask], x[mask])

        # ensure valid odd window within data length and > poly
        wl = min(wl, x_fill.size - (1 - x_fill.size % 2))
        wl = max(_odd(po + 3), wl)  # at least poly+2 and odd
        if wl > x_fill.size:  # still too big on tiny vectors
            return x

        y = savgol_filter(x_fill, window_length=wl, polyorder=po, mode='interp')
        # keep NaNs where original was NaN (optional; remove to fully inpaint)
        y[~mask] = np.nan
        return y

    T, H, W = phi.shape
    ys, xs = _roi_slices(x_range, y_range)

    # --- Wrap-safe instantaneous frequency (t-1, H, W)
    dphi_dt = _wrapsafe_dphi_dt(phi, fps)                     # rad/s
    f_inst  = (1.0 / (2.0 * np.pi)) * dphi_dt                 # Hz

    # --- Extract ROI
    A = amp[:, ys, xs]                                        # (T,   rH, rW)
    k = grad_mag[:, ys, xs]                                   # (T,   rH, rW)
    f = f_inst[:, ys, xs]                                     # (T-1, rH, rW)

    # --- Hard k gate: mask unphysically large gradients
    k = np.where(k > 0.5, np.nan, k)

    # --- Simple (unweighted) ROI means with NaN-handling
    A_t        = np.nanmean(A, axis=(1, 2))                   # (T,)
    k_t        = np.nanmean(k, axis=(1, 2))                   # (T,)
    f_t_short  = np.nanmean(f, axis=(1, 2))                   # (T-1,)

    # --- Smoothing (use ~0.15 s window by default)
    base_wl = max(3, int(round(smooth_window_s * fps)))
    wl = _odd(base_wl)

    A_t_s   = _savgol_nan(A_t,       wl, smooth_poly)
    k_t_s   = _savgol_nan(k_t,       wl, smooth_poly)
    f_t_s   = _savgol_nan(f_t_short, min(wl, max(3, (len(f_t_short)//2)*2+1)), smooth_poly)  # fit within T-1

    # --- Derive wavelength and phase speed from smoothed series
    lambda_t = np.full(T, np.nan)
    good_k   = np.isfinite(k_t_s) & (k_t_s > 0)
    lambda_t[good_k] = (2.0 * np.pi) / k_t_s[good_k]

    f_t = np.concatenate([f_t_s, [np.nan]])                   # align to length T
    v_t = f_t * lambda_t                                      # pixels/s

    return {"f_t": f_t, "k_t": k_t_s, "lambda_t": lambda_t, "v_t": v_t, "A_t": A_t_s}

def _intervals_for_trial(bfm_time, duration,
                         reaction_time=None, trial_type=None,
                         window_s=0.25):
    """
    Define fixed windows (global time, in seconds) for a trial:
      - pre-onset   : [onset - window_s, onset)
      - post-onset  : [onset, onset + window_s)
      - pre-offset  : [offset - window_s, offset)
      - post-offset : [offset, offset + window_s)
      - post-react  : [onset + ReactTime, onset + ReactTime + window_s)
                      (ONLY for trial types 'MC Hit (2)' or 'HC Hit (3)',
                       if reaction_time is finite and ≥ 0)

    Returns
    -------
    intervals : list of (stage_label, t_start_s, t_end_s)
        Times are in global movie coordinates (seconds).
    """
    onset  = float(bfm_time)
    offset = float(bfm_time + duration)
    w      = float(window_s)

    intervals = [
        ("pre_onset",   onset  - w, onset),
        ("post_onset",  onset,       onset  + w),
        ("pre_offset",  offset - w, offset),
        ("post_offset", offset,      offset + w),
    ]

    # Optional 'post_react' window: only for hits
    if trial_type in ("MC Hit (2)", "HC Hit (3)") and (reaction_time is not None):
        try:
            rt = float(reaction_time)
        except (TypeError, ValueError):
            rt = np.nan

        if np.isfinite(rt) and (rt >= 0.0):
            react_start = onset + rt
            intervals.append(
                ("post_react", react_start, react_start + w)
            )

    return intervals

def _aggregate_interval_traces(traces, i0, i1):
    """
    Aggregate 1D ROI traces over frame indices [i0, i1) using the median,
    ignoring NaNs.

    traces keys: 'A_t', 'k_t', 'lambda_t', 'v_t', 'f_t'
    (as returned by per_frame_wave_traces)

    Returns a dict with mean_* entries (but all are median by design).
    """
    def _median_vec(x):
        x = np.asarray(x)
        i0_clamped = max(0, min(i0, x.size))
        i1_clamped = max(i0_clamped, min(i1, x.size))
        if i1_clamped <= i0_clamped:
            return np.nan
        segment = x[i0_clamped:i1_clamped]
        segment = segment[np.isfinite(segment)]
        if segment.size == 0:
            return np.nan
        return float(np.median(segment))

    return {
        "mean_amp":          _median_vec(traces["A_t"]),
        "mean_k_rad_per_px": _median_vec(traces["k_t"]),
        "mean_lambda_px":    _median_vec(traces["lambda_t"]),
        "mean_speed_px_s":   _median_vec(traces["v_t"]),
        "mean_freq_hz":      _median_vec(traces["f_t"]),
    }

In [4]:
def wave_analysis(
    mouse,
    date=None,
    file=None,
    rois=None,
    trial_types=None,
    window_s=0.25,
    trial_info_root="C:/Users/Katie/Documents/Katie/Code/perception_project/perception_project/trial_info",
    smooth_window_s=0.15,
    smooth_poly=2,
    return_debug=False,
):
    """
    Fixed-window wave summary for one mouse/date, or all dates/files for that mouse.

    Stages included:
      'pre_onset', 'post_onset', 'pre_offset', 'post_offset',
      and 'post_react' (only if ReactTime < Duration for that trial).
    """
    if rois is None:
        raise ValueError("You must provide at least one ROI in the 'rois' dict.")

    # --- Load trial info and filter to mouse/(date)/(file) ---
    trial_info_path = f"{trial_info_root}/TrialInfo_{mouse}.csv"
    df_all = pd.read_csv(trial_info_path)

    mask = (df_all["AnimalCode"] == mouse)
    if date is not None:
        mask &= (df_all["Date"] == date)
    if file is not None:
        mask &= (df_all["File"] == file)
    df = df_all[mask].copy()

    if trial_types is not None:
        df = df[df["TrialType"].isin(trial_types)]

    msg = f"Found {len(df)} trials for {mouse}"
    if date is not None:
        msg += f", date {date}"
    else:
        msg += ", all dates"
    if file is not None:
        msg += f", file {file}"
    else:
        msg += ", all files"
    print(msg + ".")

    if len(df) == 0:
        if return_debug:
            return pd.DataFrame(), {}
        else:
            return pd.DataFrame()

    # Unique (date, file) combos to process
    unique_recs = (
        df[["Date", "File"]]
        .drop_duplicates()
        .sort_values(["Date", "File"])
        .to_records(index=False)
    )

    rows = []
    debug = {}

    def _movie_path(mouse, date_this, file_this):
        # Same logic as your extract_trials
        if mouse in ("cfm001mjr", "cfm002mjr"):
            return (
                "N:/GEVI_Wave/Analysis/Visual/"
                + mouse + "/20" + str(date_this) + "/" + file_this
                + "/cG_unmixed_dFF_denoised_2.h5"
            )
        else:
            return (
                "N:/GEVI_Wave/Analysis/Visual/"
                + mouse + "/20" + str(date_this) + "/" + file_this
                + "/cG_unmixed_dFF_denoised.h5"
            )

    # --- Loop over each (date, file) separately ---
    for date_this, file_this in unique_recs:
        df_file = df[(df["Date"] == date_this) & (df["File"] == file_this)]

        mov_path = _movie_path(mouse, date_this, file_this)

        with h5py.File(mov_path, "r") as mov_file:
            specs        = mov_file["specs"]
            fps          = specs["fps"][()].squeeze()
            raw_mask     = specs["extra_specs"]["mask"][()].squeeze()
            binning      = specs["binning"][()].squeeze()
            raw_outlines = specs["extra_specs"]["allenMapEdgeOutline"][()].squeeze()
            spaceorigin  = specs["spaceorigin"][()].squeeze()

            movie = mov_file["mov"]
            n_frames = movie.shape[0]
            movie_duration_s = n_frames / fps

            # Flip rules as in extract_trials
            flip_in_mask = mouse in ("cfm001mjr", "cfm002mjr", "cmm002mjr", "cmm003mjr", "rfm002mjr")
            flip_after   = mouse in ("cfm003mjr", "cfm004mjr", "cmm001mjr")

            # --- Loop over trials in this file ---
            for _, trial in df_file.iterrows():
                bfm_time        = float(trial["BFMTime"])
                duration        = float(trial["Duration"])
                reaction_time   = float(trial["ReactTime"])  # you already added this
                trial_type      = trial["TrialType"]
                trial_id        = trial["TrialID"][7:]  # strip first 7 chars, as before

                # Define intervals in global time (including optional 'post_react')
                intervals = _intervals_for_trial(
                    bfm_time,
                    duration,
                    reaction_time=reaction_time,
                    trial_type=trial_type,   
                    window_s=window_s,
                )

                # Global snippet bounds in seconds: cover all intervals we care about
                t0_global = min(t0 for _, t0, t1 in intervals)
                t1_global = max(t1 for _, t0, t1 in intervals)

                # Clamp to movie range
                t0_global = max(0.0, t0_global)
                t1_global = min(movie_duration_s, t1_global)

                if t1_global <= t0_global:
                    continue

                # Convert to frame indices for the snippet
                snippet_f0 = _time_to_idx(t0_global, fps, n_frames)
                snippet_f1 = _time_to_idx(t1_global, fps, n_frames)
                if snippet_f1 <= snippet_f0:
                    continue

                # Extract and process snippet
                snippet_clip = movie[snippet_f0:snippet_f1]  # (T_snip, y, x)

                # Apply mask & flips exactly as in extract_trials
                snippet_clip = mask_movie(snippet_clip, raw_mask, binning, flip=flip_in_mask)
                if flip_after:
                    snippet_clip = np.flip(snippet_clip, axis=1)

                T_snip = snippet_clip.shape[0]
                snippet_start_s = snippet_f0 / fps

                # Compute phase-related fields for this snippet
                amp, phi, grad_mag, gx, gy = compute_phase(snippet_clip, fps)

                # Per-ROI processing
                for roi_name, (x_range, y_range) in rois.items():
                    # Per-frame coherence and direction in ROI
                    theta_t, R_t, valid_t = roi_vectors_per_frame(
                        gx, gy, snippet_clip, fps,
                        x_range=x_range, y_range=y_range,
                        weight=True,
                    )

                    # Per-frame physical traces in ROI
                    traces = per_frame_wave_traces(
                        amp, phi, grad_mag, fps,
                        x_range=x_range, y_range=y_range,
                        smooth_window_s=smooth_window_s,
                        smooth_poly=smooth_poly,
                    )

                    # Optionally stash debug
                    if return_debug:
                        key_trial = trial_id
                        trial_dbg = debug.setdefault(key_trial, {})
                        trial_dbg[roi_name] = {
                            "theta_t": theta_t,
                            "R_t": R_t,
                            "valid_t": valid_t,
                            "traces": traces,
                            "snippet_start_s": snippet_start_s,
                            "date": date_this,
                            "file": file_this,
                        }

                    # Summarize each fixed window (including post_react if present)
                    for stage, t_start_abs, t_end_abs in intervals:
                        # Convert absolute times to snippet-relative times
                        t0_rel = t_start_abs - snippet_start_s
                        t1_rel = t_end_abs   - snippet_start_s

                        # Use mean_phase_gradient to get direction and R over the window
                        theta_deg, R_win = mean_phase_gradient(
                            gx, gy, snippet_clip, fps,
                            t_start_s=t0_rel,
                            t_end_s=t1_rel,
                            x_range=x_range,
                            y_range=y_range,
                            weight=True,
                            plot=False,
                        )

                        # Convert to frame indices within the snippet for traces
                        i0 = _time_to_idx(t0_rel, fps, T_snip)
                        i1 = _time_to_idx(t1_rel, fps, T_snip)

                        stats = _aggregate_interval_traces(traces, i0, i1)

                        rows.append({
                            "trial_id": trial_id,
                            "trial_type": trial_type,
                            "stage": stage,  # now includes 'post_react' when applicable
                            "roi_name": roi_name,
                            "interval_start_s": float(t_start_abs),
                            "interval_end_s":   float(t_end_abs),
                            "theta_deg": float(theta_deg),
                            "R": float(R_win),
                            "mean_amp": stats["mean_amp"],
                            "mean_k_rad_per_px": stats["mean_k_rad_per_px"],
                            "mean_lambda_px": stats["mean_lambda_px"],
                            "mean_speed_px_s": stats["mean_speed_px_s"],
                            "mean_freq_hz": stats["mean_freq_hz"],
                        })

    df_out = pd.DataFrame(rows, columns=[
        "trial_id","trial_type",
        "stage","roi_name",
        "interval_start_s","interval_end_s",
        "theta_deg","R",
        "mean_amp","mean_k_rad_per_px","mean_lambda_px",
        "mean_speed_px_s","mean_freq_hz",
    ])

    if return_debug:
        return df_out, debug
    else:
        return df_out
    
def wave_analysis_HCP(
    movie,
    fps,
    onset_frames,
    offset_frames,
    rois,
    window_s=0.25,
    smooth_window_s=0.15,
    smooth_poly=2,
):
    """
    Fixed-window wave summary for a single movie with known onset/offset frames 
    (designed for high contrast passive movies)

    For each trial (onset/offset pair), we define global time windows:
      - pre_onset   : [onset - window_s, onset)
      - post_onset  : [onset, onset + window_s)
      - pre_offset  : [offset - window_s, offset)
      - post_offset : [offset, offset + window_s)

    Then, for each trial and each ROI, we:
      1. Extract a movie snippet that covers all four windows.
      2. Compute phase-related fields on that snippet.
      3. Compute mean direction and coherence (R) within each window using
         mean_phase_gradient.
      4. Aggregate per-frame wave traces within each window.

    Parameters
    ----------
    movie : np.ndarray
        3D array of shape (T, y, x). Already masked and oriented as desired.
    fps : float
        Frame rate of the movie.
    onset_frames : array-like of int
        Frame indices for stimulus onset (same length as offset_frames).
    offset_frames : array-like of int
        Frame indices for stimulus offset (same length as onset_frames).
    rois : dict
        Mapping roi_name -> (x_range, y_range), same format as in the original
        summarize_fixed_windows_for_recording.
    window_s : float, optional
        Half-window length in seconds before/after onset/offset (default 0.25).
    smooth_window_s : float, optional
        Smoothing window length in seconds for per_frame_wave_traces.
    smooth_poly : int, optional
        Polynomial order for smoothing in per_frame_wave_traces.

    Returns
    -------
    df_out : pd.DataFrame
        One row per (trial, stage, ROI), with the same wave statistics fields
        as summarize_fixed_windows_for_recording. Columns include:
        ['trial_index', 'trial_id', 'trial_type', 'stage', 'roi_name',
         'interval_start_s', 'interval_end_s',
         'theta_deg', 'R',
         'mean_amp', 'mean_k_rad_per_px', 'mean_lambda_px',
         'mean_speed_px_s', 'mean_freq_hz']
    """
    if rois is None or len(rois) == 0:
        raise ValueError("You must provide at least one ROI in the 'rois' dict.")

    onset_frames = np.asarray(onset_frames)
    offset_frames = np.asarray(offset_frames)

    if onset_frames.shape != offset_frames.shape:
        raise ValueError("onset_frames and offset_frames must have the same shape.")

    n_trials = len(onset_frames)
    n_frames = movie.shape[0]
    movie_duration_s = n_frames / fps

    rows = []

    for trial_idx, (on_f, off_f) in enumerate(zip(onset_frames, offset_frames)):
        # Convert frame indices to times in seconds (global movie coordinates)
        onset_s = float(on_f) / fps
        offset_s = float(off_f) / fps
        duration = offset_s - onset_s

        # Skip pathological cases
        if duration <= 0:
            continue

        # Reuse the existing logic for defining time windows
        intervals = _intervals_for_trial(
            bfm_time=onset_s,
            duration=duration,
            reaction_time=None,
            trial_type=None,
            window_s=window_s,
        )  # returns list of (stage, t_start_s, t_end_s) in global time

        # Global snippet bounds in seconds: cover all intervals we care about
        t0_global = min(t0 for _, t0, t1 in intervals)
        t1_global = max(t1 for _, t0, t1 in intervals)

        # Clamp to movie range
        t0_global = max(0.0, t0_global)
        t1_global = min(movie_duration_s, t1_global)

        if t1_global <= t0_global:
            continue

        # Convert to frame indices for the snippet
        snippet_f0 = _time_to_idx(t0_global, fps, n_frames)
        snippet_f1 = _time_to_idx(t1_global, fps, n_frames)
        if snippet_f1 <= snippet_f0:
            continue

        # Extract snippet
        snippet_clip = movie[snippet_f0:snippet_f1]  # (T_snip, y, x)
        T_snip = snippet_clip.shape[0]
        snippet_start_s = snippet_f0 / fps

        # Compute phase-related fields for this snippet
        amp, phi, grad_mag, gx, gy = compute_phase(snippet_clip, fps)

        # Per-ROI processing
        for roi_name, (x_range, y_range) in rois.items():
            # Per-frame coherence and direction in ROI (for completeness, even if
            # you do not use theta_t / R_t directly here)
            theta_t, R_t, valid_t = roi_vectors_per_frame(
                gx, gy, snippet_clip, fps,
                x_range=x_range, y_range=y_range,
                weight=True,
            )

            # Per-frame physical traces in ROI
            traces = per_frame_wave_traces(
                amp, phi, grad_mag, fps,
                x_range=x_range, y_range=y_range,
                smooth_window_s=smooth_window_s,
                smooth_poly=smooth_poly,
            )

            # Summarize each fixed window
            for stage, t_start_abs, t_end_abs in intervals:
                # Convert absolute times to snippet-relative times
                t0_rel = t_start_abs - snippet_start_s
                t1_rel = t_end_abs   - snippet_start_s

                # Check if this interval overlaps the snippet at all
                if (t1_rel <= 0.0) or (t0_rel >= T_snip / fps):
                    continue

                # Use mean_phase_gradient to get direction and R over the window
                theta_deg, R_win = mean_phase_gradient(
                    gx, gy, snippet_clip, fps,
                    t_start_s=t0_rel,
                    t_end_s=t1_rel,
                    x_range=x_range,
                    y_range=y_range,
                    weight=True,
                    plot=False,
                )

                # Convert to frame indices within the snippet for traces
                i0 = _time_to_idx(t0_rel, fps, T_snip)
                i1 = _time_to_idx(t1_rel, fps, T_snip)
                if i1 <= i0:
                    continue

                stats = _aggregate_interval_traces(traces, i0, i1)

                rows.append({
                    "trial_index": trial_idx,
                    "trial_id": f"trial_{trial_idx}",
                    "trial_type": None,  # no trial_type for these data
                    "stage": stage,
                    "roi_name": roi_name,
                    "interval_start_s": float(t_start_abs),
                    "interval_end_s":   float(t_end_abs),
                    "theta_deg": float(theta_deg),
                    "R": float(R_win),
                    "mean_amp": stats["mean_amp"],
                    "mean_k_rad_per_px": stats["mean_k_rad_per_px"],
                    "mean_lambda_px": stats["mean_lambda_px"],
                    "mean_speed_px_s": stats["mean_speed_px_s"],
                    "mean_freq_hz": stats["mean_freq_hz"],
                })

    df_out = pd.DataFrame(rows, columns=[
        "trial_index", "trial_id", "trial_type",
        "stage", "roi_name",
        "interval_start_s", "interval_end_s",
        "theta_deg", "R",
        "mean_amp", "mean_k_rad_per_px", "mean_lambda_px",
        "mean_speed_px_s", "mean_freq_hz",
    ])

    return df_out

In [ ]:
rois = {
    "SSC": ((95,115), (30,50)),
    "V1":  ((90,110), (150,170)),
    "M2":  ((30,50), (15,35))
}

df_windows = wave_analysis(
    mouse="cfm002mjr",
    date=None,
    file=None,
    rois=rois,
    trial_types = ['HC Hit (3)', 'HC No Report (9)', 'Incorrect Reject (6)', 'MC Hit (2)', 'MC Miss (5)', 'MC No Report (8)', 'Correct Rejection (4)', 'LC No Report (7)'],
    window_s=0.25,
    return_debug=False,
)

Found 1926 trials for cfm002mjr, all dates, all files.


C:\Users\Katie\AppData\Local\Temp\ipykernel_24884\1427452551.py:197: RuntimeWarning: invalid value encountered in scalar divide
  mx = np.sum(wx) / denom
C:\Users\Katie\AppData\Local\Temp\ipykernel_24884\1427452551.py:198: RuntimeWarning: invalid value encountered in scalar divide
  my = np.sum(wy) / denom


In [47]:
df_windows.to_csv('cfm002_wave_analysis_v2.csv')
df_windows

,trial_id,trial_type,stage,roi_name,interval_start_s,interval_end_s,theta_deg,R,mean_amp,mean_k_rad_per_px,mean_lambda_px,mean_speed_px_s,mean_freq_hz
0,cfm002mjr/20240510/meas00/trial002,MC No Report (8),pre_onset,SSC,1.053997,1.303997,158.930357,0.537901,0.008226,0.070872,88.655083,389.537939,3.746307
1,cfm002mjr/20240510/meas00/trial002,MC No Report (8),post_onset,SSC,1.303997,1.553997,-94.008496,0.232829,0.006472,0.093813,66.975574,284.167119,3.727414
2,cfm002mjr/20240510/meas00/trial002,MC No Report (8),pre_offset,SSC,1.569997,1.819997,-112.943731,0.258335,0.004808,0.107952,58.203680,403.431596,6.740885
3,cfm002mjr/20240510/meas00/trial002,MC No Report (8),post_offset,SSC,1.819997,2.069997,-163.661135,0.123156,0.004722,0.109201,57.537748,473.028238,9.544017
4,cfm002mjr/20240510/meas00/trial002,MC No Report (8),pre_onset,V1,1.053997,1.303997,-151.818817,0.176184,0.003016,0.145725,43.116815,194.365251,4.664844
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23098,cfm002mjr/20240519/meas02/trial139,MC No Report (8),post_offset,V1,601.718069,601.968069,NaN,NaN,0.001448,0.192750,32.597652,253.439984,7.772970
23099,cfm002mjr/20240519/meas02/trial139,MC No Report (8),pre_onset,M2,600.933069,601.183069,169.403824,0.868739,0.020157,0.043107,145.758286,334.460560,2.463953
23100,cfm002mjr/20240519/meas02/trial139,MC No Report (8),post_onset,M2,601.183069,601.433069,-158.210801,0.756813,0.013953,0.050507,124.402676,206.763654,1.438431
23101,cfm002mjr/20240519/meas02/trial139,MC No Report (8),pre_offset,M2,601.468069,601.718069,-46.198485,0.314744,0.011372,0.056418,111.367564,195.561176,2.185417


In [61]:
path_noisy = "N:/GEVI_Wave/Analysis/Visual/cmm002mjr/20231208/meas00/cG_unmixed_dFF.h5"
path_denoised = "N:/GEVI_Wave/Analysis/Visual/cmm002mjr/20231208/meas00/_denoised.h5"
movie_denoised, fps, onsets, offsets = load_and_mask(path_denoised, path_specs=path_noisy)

In [66]:
rois = {
    "SSC": ((95,115), (30,50)),
    "V1":  ((100,120), (180,200)),
    "M2":  ((40,60), (30,50))
}

df_hcp = wave_analysis_HCP(movie_denoised,
                           fps,
                           onsets,
                           offsets[1:],
                           rois)

In [68]:
df_hcp.to_csv('cmm002_wave_analysis_hcp.csv')

In [5]:
path_noisy = "N:/GEVI_Wave/Analysis/Visual/cfm001mjr/20231208/meas00/cG_unmixed_dFF.h5"
path_denoised = "N:/GEVI_Wave/Analysis/Visual/cfm001mjr/20231208/meas00/denoised.h5"
movie_denoised, fps, onsets, offsets = load_and_mask(path_denoised, path_specs=path_noisy)

In [7]:
rois = {
    "SSC": ((85,105), (35,55)),
    "V1":  ((95,115), (160,180)),
    "M2":  ((35,55), (25,45))
}

df_hcp = wave_analysis_HCP(movie_denoised,
                           fps,
                           onsets,
                           offsets,
                           rois)

In [9]:
df_hcp.to_csv('cfm001_wave_analysis_hcp.csv')

In [19]:
path_noisy = "N:/GEVI_Wave/Analysis/Visual/cmm001mjr/20231208/meas00/cG_unmixed_dFF.h5"
path_denoised = "N:/GEVI_Wave/Analysis/Visual/cmm001mjr/20231208/meas00/denoised.h5"
movie_denoised, fps, onsets, offsets = load_and_mask(path_denoised, path_specs=path_noisy)

KeyboardInterrupt: 

In [16]:
rois = {
    "SSC": ((100,120), (45,65)),
    "V1":  ((100,120), (170,190)),
    "M2":  ((43,63), (43,63))
}

df_hcp = wave_analysis_HCP(movie_denoised,
                           fps,
                           onsets[:-1], 
                           offsets[1:],
                           rois)

In [18]:
df_hcp.to_csv('cmm001_wave_analysis_hcp.csv')